# 07 面试复盘：把代码细节讲成工程闭环

前面 6 本解决了底层问题。这一本把它们组织成面试能讲的工程故事。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap, re

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/qwen lorar sft/qwen-local-lora-sft-dpo")

print("项目根目录:", PROJECT_ROOT)
print("学习原则: 本 notebook 默认只读项目文件；所有真实推理/训练开关默认 False。")

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=None):
    lines = read_text(rel).splitlines()
    if end is None:
        end = len(lines)
    print(f"--- {rel}:{start}-{end} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = []
    for idx, line in enumerate(lines, start=1):
        if keyword in line:
            hits.append(idx)
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        start = max(1, hit - context)
        end = min(len(lines), hit + context)
        show_file(rel, start, end)
        print()

def read_jsonl(rel, n=3):
    rows = []
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
            if len(rows) >= n:
                break
    return rows

def count_jsonl(rel):
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

## 1. 30 秒版本

我在本地 RTX 4060 Laptop GPU 上跑通了 Qwen2.5-0.5B 的 LoRA SFT/DPO 后训练链路。项目先用 public SFT 证明 Qwen 加载、数据转换、LoRA 训练、adapter 保存加载和固定 prompt 对比都能跑，再用 badcase 驱动自采集技术数据和 custom SFT。custom SFT 把 LoRA/SFT/DPO 概念类 prompt 提升到 7/8。后续我跑通了 DPO，v6 是最好的 DPO artifact，但因为 prompt 7 行为门禁仍失败，所以最终推荐 checkpoint 仍是 custom SFT v3。核心结论是：loss、preference accuracy 和真实行为验收必须分开看。

## 2. 从代码角度串起来

| 问题 | 代码位置 | 面试解释 |
|---|---|---|
| Qwen 怎么加载？ | `scripts/infer.py` 的 `from_pretrained` | Hugging Face 从本地 cache 或网络加载 tokenizer/model |
| 聊天怎么变 token？ | `apply_chat_template` | 使用 Qwen 自带聊天模板，避免手写格式 |
| SFT 学哪里？ | `ChatSFTDataset.encode` | prompt labels 设 -100，只学 assistant 答案 |
| LoRA 怎么实现？ | `LoraConfig` + `get_peft_model` | 冻结 base，大部分只训练 adapter |
| adapter 怎么复用？ | `PeftModel.from_pretrained` | base Qwen + LoRA 小补丁组合推理 |
| 怎么判断好坏？ | `compare_*` + `score_*` | 固定 prompt 门禁，防止只看 loss |
| DPO 怎么做？ | `read_dpo_rows` + `DPOTrainer` | 用 chosen/rejected 偏好对训练 |

## 3. 高频问题：为什么 public SFT 失败也有价值？

因为 public SFT 的目的不是直接修所有目标概念，而是证明工程链路能跑：数据格式、训练、保存、加载、对比全部闭环。它没有修好 LoRA/SFT/DPO，反而成为 Stage 2B 自采集技术数据的证据。

## 4. 高频问题：为什么不只看 loss？

loss 是平均训练信号，不等于目标 prompt 的真实行为。项目里 public-SFT 能训练成功但概念仍错；后续某些 patch loss 更好却让旧 prompt 回归。所以项目用固定 prompt、badcase review 和旧能力回归检查做行为门禁。

## 5. 高频问题：DPO 到底算不算做了？

算做了，而且跑通了多轮：tiny DPO、candidate DPO、larger naive DPO v6、v7/v8 修复尝试。硬件不是唯一瓶颈，preference accuracy 也能很好看。但没有一个 DPO adapter 完全通过行为门禁，所以它们是有价值的实验 artifact，不是默认推荐 checkpoint。

## 6. 你必须背熟的两个路径

In [ ]:
for rel in ["outputs/sft_lora_qwen05b_custom_v3_from_v1_patch", "outputs/dpo_lora_qwen05b_naive_v6"]:
    print(rel, "OK" if (PROJECT_ROOT / rel).exists() else "MISSING")

## 7. 最终项目价值

这个项目最像工程项目的地方，不是“我训练了一个 LoRA”，而是：

```text
base badcase
  -> public baseline 验证链路
  -> custom data 修目标问题
  -> fixed prompt 回归测试
  -> DPO 偏好优化实验
  -> 指标好但行为不过关就拒绝
```